In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 11. RNNとLSTM

> **学習目標**
> - RNN
> - (long-term dependency) 問題
> - LSTM

## 11.1 データ 問題

 ベクトル, . ?

- **RNN**:
- **CNN (1D)**:
- **Transformer (Ch 14+)**: Attention

 RNN/LSTM . .

## 11.2 RNN

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$$

- $\mathbf{x}_t \in \mathbb{R}^d$: 時間 $t$
- $\mathbf{h}_t \in \mathbb{R}^h$: 時間 $t$ 
- $W_h \in \mathbb{R}^{h \times h}$, $W_x \in \mathbb{R}^{h \times d}$:

: ** 時間 ** ( ).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#  RNN  (NumPy)
class RNNCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        self.Wx = rng.standard_normal((input_dim, hidden_dim)) * 0.1
        self.Wh = rng.standard_normal((hidden_dim, hidden_dim)) * 0.1
        self.b = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        """x_seq: (T, input_dim) -> h_seq: (T, hidden_dim)"""
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        h_seq = []
        for t in range(T):
            h = np.tanh(x_seq[t] @ self.Wx + h @ self.Wh + self.b)
            h_seq.append(h.copy())
        return np.array(h_seq)

    def last_hidden(self, x_seq):
        return self.forward(x_seq)[-1]

# Test
rnn = RNNCell(input_dim=4, hidden_dim=8)
x_seq = np.random.randn(10, 4)  # Length 10 
h_seq = rnn.forward(x_seq)
print(f" : {x_seq.shape}")
print(f"  : {h_seq.shape}")
print(f"  : {h_seq[-1].round(4)}")


## 11.3 BPTT (Backpropagation Through Time)

RNN バックプロパゲーション = 時間 計算 グラフ バックプロパゲーション.

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T}$$

$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \mathrm{diag}(1 - \mathbf{h}_t^2) W_h$ (tanh 導関数 ).

問題: $\mathrm{diag}(1 - \mathbf{h}_t^2) \leq 1$ $W_h$ 値 値 1 ** **, ****.


In [ ]:
#  / 
def simulate_rnn_gradient(T, Wh_norm, seed=0):
    """T Step RNN h_0  Gradient Magnitude."""
    rng = np.random.default_rng(seed)
    h = rng.standard_normal(8)
    grad = np.ones(8)  # dL/dh_T
    for t in range(T):
        # Wh  Matrix norm
        Wh = np.eye(8) * Wh_norm
        h_prev = rng.standard_normal(8)
        h = np.tanh(h_prev @ Wh)
        # Backward Pass: grad = grad @ Wh @ diag(1 - h^2)
        diag = 1 - h**2
        grad = grad @ Wh * diag
    return np.linalg.norm(grad)

print("RNN  / :")
print(f"{'T':>5} | {'||Wh||=0.5 ()':>20} | {'||Wh||=1.0':>15} | {'||Wh||=1.5 ()':>20}")
print("-" * 65)
for T in [5, 10, 20, 50, 100]:
    g_small = simulate_rnn_gradient(T, 0.5)
    g_one = simulate_rnn_gradient(T, 1.0)
    g_big = simulate_rnn_gradient(T, 1.5)
    print(f"{T:>5} | {g_small:>20.4e} | {g_one:>15.4e} | {g_big:>20.4e}")

print("\n=> ||Wh||<1の場合 Gradient  Decrease ()")
print("=> ||Wh||>1の場合 Gradient  Increase ()")


## 11.4 LSTM —

LSTM (1997, Hochreiter & Schmidhuber) .

$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate)}$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate)}$$
$$\tilde{\mathbf{c}}_t = \tanh(W_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(candidate)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state)}$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state)}$$

: **cell state $\mathbf{c}_t$ ** → .


In [ ]:
# LSTM   (NumPy)
class LSTMCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        # 4 : forget, input, candidate, output
        self.W = rng.standard_normal((input_dim + hidden_dim, 4 * hidden_dim)) * 0.1
        self.b = np.zeros(4 * hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        h_seq, c_seq = [], []
        for t in range(T):
            # : [h, x]
            hx = np.concatenate([h, x_seq[t]])
            gates = hx @ self.W + self.b
            f, i, g, o = np.split(gates, 4)
            f = 1 / (1 + np.exp(-f))  # sigmoid
            i = 1 / (1 + np.exp(-i))
            g = np.tanh(g)
            o = 1 / (1 + np.exp(-o))
            c = f * c + i * g  # cell state ( !)
            h = o * np.tanh(c)
            h_seq.append(h.copy()); c_seq.append(c.copy())
        return np.array(h_seq), np.array(c_seq)

# 
lstm = LSTMCell(input_dim=4, hidden_dim=8)
h_seq, c_seq = lstm.forward(x_seq)
print(f"LSTM Output: h_seq {h_seq.shape}, c_seq {c_seq.shape}")
print(f" h: {h_seq[-1].round(4)}")
print(f" c: {c_seq[-1].round(4)}")


## 11.5 モデル

 RNN/LSTM .


In [ ]:
#   LSTM  モデル (PyTorch)
import torch
import torch.nn as nn

# Tiny Shakespeare  
from llm_math.data import load_tiny_shakespeare
text = load_tiny_shakespeare(max_chars=20000)
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print(f" Length: {len(text)}, Vocabulary Size: {vocab_size}")
print(f": {chars[:20]}...")

# データ 
seq_len = 50
data = np.array([char_to_idx[c] for c in text])

def make_batch(data, seq_len, batch_size=32):
    """ Batch Generation."""
    n = len(data) - seq_len - 1
    starts = np.random.randint(0, n, batch_size)
    X = np.array([data[s:s+seq_len] for s in starts])
    y = np.array([data[s+1:s+1+seq_len] for s in starts])
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Model
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, n_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x)  # (B, T, E)
        out, hidden = self.lstm(emb, hidden)  # (B, T, H)
        logits = self.fc(out)  # (B, T, V)
        return logits, hidden

model = CharLSTM(vocab_size, embed_dim=32, hidden_dim=64)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nモデル  : {n_params:,}")


In [ ]:
# 学習
import time

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.003)

n_steps = 300
losses = []
t0 = time.time()
for step in range(n_steps):
    X, y = make_batch(data, seq_len, batch_size=32)
    opt.zero_grad()
    logits, _ = model(X)
    # (B*T, V) vs (B*T,)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    opt.step()
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss={loss.item():.4f}")
print(f"\nTraining Time: {time.time() - t0:.1f}")


In [ ]:
#  
def generate(model, seed_text, length=200, temperature=0.8):
    model.eval()
    chars_idx = [char_to_idx[c] for c in seed_text]
    hidden = None
    with torch.no_grad():
        x = torch.tensor([chars_idx], dtype=torch.long)
        for _ in range(length):
            logits, hidden = model(x, hidden)
            #  時間 
            logit = logits[0, -1] / temperature
            probs = torch.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            chars_idx.append(next_idx)
            #    
            x = torch.tensor([[next_idx]], dtype=torch.long)
    return ''.join(idx_to_char[i] for i in chars_idx)

print("=== 生成  (Training ) ===\n")
print(generate(model, "First Citizen:\n", length=300))


In [ ]:
# 学習 
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, color='blue')
#  Mean
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2, label='Mean')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Character-level LSTM Learning Curve')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch11_char_lstm.png', dpi=100, bbox_inches='tight')
plt.show()


## 11.6 GRU — LSTM

GRU (2014, Cho et al.) 2 :

$$\mathbf{z}_t = \sigma(W_z [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(update gate)}$$
$$\mathbf{r}_t = \sigma(W_r [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(reset gate)}$$
$$\tilde{\mathbf{h}}_t = \tanh(W [\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t])$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

cell state . 性能 LSTM .


## 11.7 Attention — RNN

RNN :
1. ** **: →
2. ** **: LSTM度
3. ** **: ベクトル

**Attention(attention)** . (Bahdanau et al., 2014)

 (Ch 14) .

## 11.8 要点

| | | |
|---|---|---|
| RNN | $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t)$ | , / |
| LSTM | + cell state  | |
| GRU | 2  | LSTM |
| BPTT | 時間 バックプロパゲーション | $\prod \frac{\partial h_t}{\partial h_{t-1}}$ |

## 演習問題

1. RNN "hello world" 学習 .
2. LSTM GRU データ 学習 速度 比較.
3. シーケンス長 10, 50, 100 RNN LSTM 比較.
4. RNN .
5. LSTM temperature 0.3, 0.8, 1.5 比較.

> 解答: `solutions/ch11_solutions.ipynb`
